# **GROUP DNA** : The WhatsApp Chat Analyzer

**Name**: [Abhay Pratap Singh]

**Roll Number**: [17729]

**Batch**: [Your Batch]
# Date: 20-08-2026

# Feature 1:  THE PARSER


In [ ]:
from datetime import datetime, timedelta
import numpy as np
import string

file_path = 'hostel_bois.txt'

with open(file_path, 'r', encoding='utf-8') as f:
    lines = f.readlines()

messages = []
sys_msgs = 0
media_msgs = 0
deleted_msgs = 0

for line in lines:
    line = line.strip()
    if not line:
        continue

    parts = line.split(' - ', 1)
    if len(parts) < 2:
        sys_msgs += 1
        continue

    timestamp_str, rest = parts
    try:
        # ASSUMING FORMAT DD/MM/YY, HH:MM
        dt = datetime.strptime(timestamp_str, '%d/%m/%y, %H:%M')
    except ValueError:
        try:
            # FALL BACK FOR 4 DIGIT YEAR (DD/MM/YYYY) JUST IN CASE
            dt = datetime.strptime(timestamp_str, '%d/%m/%Y, %H:%M')
        except ValueError:
            sys_msgs += 1
            continue

    sender_parts = rest.split(': ', 1)
    if len(sender_parts) < 2:
        sys_msgs += 1
        continue

    sender, msg_text = sender_parts

    is_media = False
    is_deleted = False

    if msg_text == '<Media omitted>':
        media_msgs += 1
        is_media = True
    elif msg_text == 'This message was deleted':
        deleted_msgs += 1
        is_deleted = True

    messages.append({
        'dt': dt,
        'sender': sender,
        'text': msg_text,
        'is_media': is_media,
        'is_deleted': is_deleted
    })

print(f"Successfully parsed {len(messages)} messages.")
print(f"Skipped {sys_msgs} system messages, counted {media_msgs} media omitted and {deleted_msgs} deleted messages.")
print("First 5 messages:")
for m in messages[:5]: print(m)



## Feature 2: GROUP OVERVIEW


In [ ]:
participants = set(m['sender'] for m in messages)
start_date = messages[0]['dt']
end_date = messages[-1]['dt']
total_days = (end_date - start_date).days + 1

msg_counts = {p: 0 for p in participants}
for m in messages:
    msg_counts[m['sender']] += 1

sorted_counts = sorted(msg_counts.items(), key=lambda x: x[1], reverse=True)

print("============================================================")
print("    GROUP OVERVIEW   ")
print("============================================================")
print(f" Group       : Hostel Bois 4ever")
print(f" Period      : {start_date.strftime('%d %B %Y')} to {end_date.strftime('%d %B %Y')} ({total_days} days)")
print(f" Total messages : {len(messages):,}")
print(f" Participants   : {len(participants)}")
print("MESSAGES PER PERSON")
for person, count in sorted_counts:
    pct = (count / len(messages)) * 100
    print(f" {person:<10} : {count:>4} ({pct:>4.1f}%)")



## Feature 3: MOST ACTIVE DAY AND HOUR


In [ ]:
daily_counts = {}
hourly_counts = {}

for i in messages:
    d = i['dt'].date()
    h = i['dt'].hour
    daily_counts[d] = daily_counts.get(d, 0) + 1
    hourly_counts[h] = hourly_counts.get(h, 0) + 1

busiest_day = max(daily_counts, key=daily_counts.get)
busiest_hour = max(hourly_counts, key=hourly_counts.get)
avg_per_day_busiest_hour = hourly_counts[busiest_hour] / total_days

print(f"The Busiest day  : {busiest_day.strftime('%d %B %Y')} ({daily_counts[busiest_day]} messages)")
print(f"The Busiest hour : {busiest_hour:02d}:00 - {busiest_hour+1:02d}:00  (avg {int(avg_per_day_busiest_hour)} messages per day)")



## Feature 4: THE ACTIVITY HEATMAP (NumPy)


In [ ]:
participant_list = [p for p, _ in sorted_counts]
heatmap = np.zeros((len(participant_list), 24), dtype=int)

for n in messages:
    p_idx = participant_list.index(n['sender'])
    h_idx = n['dt'].hour
    heatmap[p_idx, h_idx] += 1

print("THE ACTIVITY HEATMAP (messages by hour)")
# WE WILL PRINT 24 COLUMNS, HEADERS EVERY 3 HOURS
print("       " + "".join([f"{h:02d} " if h % 3 == 0 else "   " for h in range(24)]))

for i, p in enumerate(participant_list):
    row = heatmap[i]
    max_val = max(row)
    if max_val == 0: max_val = 1

    blocks = []
    for val in row:
        pct = val / max_val
        if pct == 0:
            blocks.append('. ')
        elif pct <= 0.25:
            blocks.append('. ')
        elif pct <= 0.50:
            blocks.append('░ ')
        elif pct <= 0.75:
            blocks.append('▒ ')
        else:
            blocks.append('█ ')

    print(f" {p:<5} " + "".join(blocks))



## Feature 5: TOP WORDS


In [ ]:
stop_words = {'i', 'is', 'the', 'a', 'and', 'or', 'to', 'of', 'in', 'on', 'for', 'it', 'my', 'me', 'you', 'that', 'this', 'hai', 'ki', 'se', 'ko', 'bhi', 'na', 'toh', 'ke', 'ka', 'ye', 'kya'}
word_counts = {}

for n in messages:
    if n['is_media'] or n['is_deleted']:
        continue

    text = n['text'].lower()
    for p in string.punctuation:
        text = text.replace(p, ' ')

    words = text.split()
    for w in words:
        if w not in stop_words and len(w) > 1:
            word_counts[w] = word_counts.get(w, 0) + 1

top_words = sorted(word_counts.items(), key=lambda x: x[1], reverse=True)[:10]

print("THIS IS GROUP'S FAVOURITE WORDS")
for w, count in top_words:

    # 20 BLOCKS MAX LENGTH
    bar_len = int((count / top_words[0][1]) * 20)
    bar = '█' * bar_len
    print(f" {w:<10} {bar:<20} {count}")



## Feature 6: RESPONSE SPEED AND THE SILENT STREAKS


In [ ]:
response_times = {p: [] for p in participant_list}
silent_streaks = {p: 0 for p in participant_list}

last_sender = None
last_dt = None

# FOR RESPONSE TIME

for j in messages:
    if last_sender and last_sender != j['sender']:
        gap = (j['dt'] - last_dt).total_seconds()
        response_times[j['sender']].append(gap)
    last_sender = j['sender']
    last_dt = j['dt']

avg_responses = {}
for p, gaps in response_times.items():
    if gaps:
        avg_responses[p] = sum(gaps) / len(gaps)
    else:
        avg_responses[p] = 0

# FOR SILENT STREAKS

date_set = {m['dt'].date() for m in messages}
all_dates = sorted(list(date_set))

for p in participant_list:
    max_streak = 0
    current_streak = 0
    p_dates = {m['dt'].date() for m in messages if m['sender'] == p}

    for d in all_dates:
        if d not in p_dates:
            current_streak += 1
            if current_streak > max_streak:
                max_streak = current_streak
        else:
            current_streak = 0
    silent_streaks[p] = max_streak

fastest = min([p for p in avg_responses.items() if p[1] > 0], key=lambda x: x[1])
slowest = max([p for p in avg_responses.items() if p[1] > 0], key=lambda x: x[1])

print(" THE RESPONSE PATTERNS")
print(f"Fastest replier : {fastest[0]} (avg {fastest[1]/60:.1f} minutes)")
print(f"Slowest replier : {slowest[0]} (avg {slowest[1]/3600:.1f} hours)")

print("LONGEST SILENT STREAKS")
for p, streak in sorted(silent_streaks.items(), key=lambda x: x[1], reverse=True):
    print(f" {p:<10} : {streak} days")



## Feature 7: PERSONALITY ARCHETYPE DETECTION


In [ ]:
archetypes = {p: [] for p in participant_list}

# 1. THE SPAMMER : Avg consecutive message burst > 3
bursts = {p: [] for p in participant_list}
current_burst = 0
curr_sender = None

for m in messages:
    if m['sender'] == curr_sender:
        current_burst += 1
    else:
        if curr_sender:
            bursts[curr_sender].append(current_burst)
        curr_sender = m['sender']
        current_burst = 1
if curr_sender: bursts[curr_sender].append(current_burst)

spammer_scores = {p: (sum(b)/len(b) if b else 0) for p, b in bursts.items()}

# 2. THE QUESTION MASTER : end with ?
qm_scores = {p: 0 for p in participant_list}
for k in messages:
    if not k['is_media'] and not k['is_deleted']:
        if k['text'].strip().endswith('?'):
            qm_scores[k['sender']] += 1
qm_pct = {p: (qm_scores[p] / msg_counts[p] * 100) for p in participant_list}


# 3. THE NIGHT OWL : > 60% messages between 23-04
owl_scores = {p: 0 for p in participant_list}
for i, p in enumerate(participant_list):
    night_msgs = sum(heatmap[i, 23:]) + sum(heatmap[i, 0:5])
    total_msgs = sum(heatmap[i, :])
    owl_scores[p] = (night_msgs / total_msgs * 100) if total_msgs > 0 else 0

# 4. THE STORYTELLER : Avg words per message > 30
words_per_msg = {p: [] for p in participant_list}
for m in messages:
    if not m['is_media'] and not m['is_deleted']:
        words_per_msg[m['sender']].append(len(m['text'].split()))
storyteller_scores = {p: (sum(w)/len(w) if w else 0) for p, w in words_per_msg.items()}

# 5. THE DRAMA QUEEN : > 30% messages all-caps or 2+ !
drama_scores = {p: 0 for p in participant_list}
for j in messages:
    if not j['is_media'] and not j['is_deleted']:
        t = j['text']
        if (t.isupper() and len(t) > 3) or t.count('!') >= 2:
            drama_scores[j['sender']] += 1
drama_pct = {p: (drama_scores[p] / msg_counts[p] * 100) for p in participant_list}

# 6. THE GHOST : Silent > 60% of days
ghost_scores = {p: 0 for p in participant_list}
for p in participant_list:
    days_active = len({m['dt'].date() for m in messages if m['sender'] == p})
    days_silent = total_days - days_active
    ghost_scores[p] = (days_silent / total_days * 100)

# 7. THE COMEDIAN : 'lol', 'lmao', etc.
haha_words = ['lol', 'lmao', 'haha', 'rofl', 'lmfao']
comedian_scores = {p: 0 for p in participant_list}
for m in messages:
    if not m['is_media'] and not m['is_deleted']:
        t = m['text'].lower()
        if any(hw in t for hw in haha_words):
            comedian_scores[m['sender']] += 1
comedian_pct = {p: (comedian_scores[p] / msg_counts[p] * 100) for p in participant_list}

# 8. THE GROUP MOM : caring keywords
caring_words = ['okay', 'safe', 'eat', 'sleep', 'take care', 'are you', 'please', 'reminder', 'drink water', "don't forget"]
mom_scores = {p: 0 for p in participant_list}
for n in messages:
    if not n['is_media'] and not n['is_deleted']:
        t = n['text'].lower()
        if any(cw in t for cw in caring_words):
            mom_scores[n['sender']] += 1

# ASSIGN ARCHETYPES
final_archetypes = {}
for p in participant_list:
    scores = {
        'THE SPAMMER': spammer_scores[p] if spammer_scores[p] > 3 else 0,
        'THE GROUP MOM': mom_scores[p],
        'THE NIGHT OWL': owl_scores[p] if owl_scores[p] > 60 else 0,
        'THE STORYTELLER': storyteller_scores[p] if storyteller_scores[p] > 30 else 0,
        'THE DRAMA QUEEN': drama_pct[p] if drama_pct[p] > 30 else 0,
        'THE GHOST': ghost_scores[p] if ghost_scores[p] > 60 else 0,
        'THE COMEDIAN': comedian_pct[p],
        'THE QUESTION MASTER': qm_pct[p] if qm_pct[p] > 25 else 0
    }

    # WE WILL PICK THE MAX SCORE. IF ALL 0, PICK A TIEBREAKER
    best_arch = max(scores, key=scores.get)

    # SIMPLE FORMATTING FOR THE REASON
    reason = f"score: {scores[best_arch]:.1f}"
    if best_arch == 'THE SPAMMER': reason = f"avg {scores[best_arch]:.1f} msgs in a row"
    elif best_arch == 'THE NIGHT OWL': reason = f"{scores[best_arch]:.1f}% msgs at night"
    elif best_arch == 'THE STORYTELLER': reason = f"avg {scores[best_arch]:.1f} words per msg"
    elif best_arch == 'THE DRAMA QUEEN': reason = f"{scores[best_arch]:.1f}% ALL-CAPS/!"
    elif best_arch == 'THE GHOST': reason = f"silent {scores[best_arch]:.1f}% of days"

    final_archetypes[p] = (best_arch, reason)

print("PERSONALITY ARCHETYPES")
for p, (arch, reason) in final_archetypes.items():
    print(f" {p:<10} -> {arch:<18} ({reason})")

